# Flux reconciliation for the ECOMICS experimental reaction fluxes

In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
import pyomo.environ as pyo
from pyomo.opt import SolverStatus, TerminationCondition
import pandas as pd
from typing import Dict, List, Tuple, Optional
import warnings
import os
import sys
import cobra
from cobra.io import load_model

**QUESTIONS**


- To set the S matrix, could I use the iML1515 S matrix (2700 rxns and 1200 mets)? Or does it have to stick to the measured fluxes 140 reactions and mets?
- For the bounds do I need to 
- Will look if there are stedevs in original studies. If I don't find any, could I average the fluxes across the same conditions and use the resulting stdevs?

# Small network

In [3]:
# Network: Glucose_ext --(uptake)--> Glucose_int --(glycolysis)--> Pyruvate_int
#                                                                      |
#                                                         |------------|------------|
#                                                         v                         v
#                                                   Lactate_ext                Biomass

### 1. Model Inputs


In [30]:
# S matrix 

# Columns: reactions
#   1. Glucose_uptake
#   2. Pyr_formation
#   3. Lactate_formation
#   4. Biomass_formation

# Rows: metabolites 
#   1. Glucose_internal
#   2. Pyruvate_internal

S = np.array([
    [ 1, -1,  0,  0],  # Glucose_internal: +1 from uptake, -1 to pyruvate formation
    [ 0,  1, -1, -1],  # Pyruvate_internal: +1 from glucose, -1 to lactate, -1 to biomass  
])

metabolite_names = ['Glucose_internal', 'Pyruvate_internal']
reaction_names = ['Glucose_uptake', 'Pyr_formation', 'Lactate_formation', 'Biomass_formation']

n_metabolites = len(metabolite_names)
n_reactions = len(reaction_names)

# Reaction fluxes
# Note: all fluxes should be positive for forward reactions
# The S matrix handles the directionality (consumption vs production)
measurements = {
    'Glucose_uptake': 10,      # Positive: rate of glucose consumption
    'Pyr_formation': 10,       # Should equal glucose uptake due to mass balance
    'Lactate_formation': 6,    # Part of pyruvate goes to lactate
    'Biomass_formation': 4     # Part of pyruvate goes to biomass
}

# Reaction fluxes stdevs
measurement_stdevs = {
    'Glucose_uptake': 0.5,
    'Pyr_formation': 0.1,
    'Lactate_formation': 0.6,
    'Biomass_formation': 0.4
}

# # Fixed fluxes
# fixed_fluxes = {
#     'Glucose_uptake': 10.0  # Fix glucose uptake rate
# }

# Flux bounds - all reactions are forward (positive)
flux_bounds = {
    'Glucose_uptake': (0, 20),    # Forward only (glucose consumption)
    'Pyr_formation': (0, 20),     # Forward only
    'Lactate_formation': (0, 15), # Forward only
    'Biomass_formation': (0, 10)  # Forward only
}

### 2. Check mass balance consistency

Mass balance constraints from S matrix:
1. Glucose_internal: v_glucose_uptake - v_pyr_formation = 0
   => v_glucose_uptake = v_pyr_formation
2. Pyruvate_internal: v_pyr_formation - v_lactate_formation - v_biomass_formation = 0
   => v_pyr_formation = v_lactate_formation + v_biomass_formation

Note: Model weights section removed - not needed for simple flux reconciliation. We're just minimizing deviations from measurements, not optimizing for any particular objective

In [31]:
# Vector of initial weights (zeros). Each entry corresponds to a reaction
objective_weights = np.zeros(n_reactions)

# Assign weight of 1.0 to reaction of objective function
objective_reaction = 'Biomass_formation'
reaction_idx = reaction_names.index(objective_reaction)
objective_weights[reaction_idx] = 1.0

In [32]:
# Check mass balance consistency of measurements
print("Current measurements:")
for reaction, flux in measurements.items():
    print(f"  {reaction}: {flux}")

# Check constraints
glucose_uptake = measurements['Glucose_uptake']
pyr_formation = measurements['Pyr_formation'] 
lactate_formation = measurements['Lactate_formation']
biomass_formation = measurements['Biomass_formation']

print(f"\nConstraint 1 check:")
print(f"  Glucose uptake: {glucose_uptake}")
print(f"  Pyruvate formation: {pyr_formation}")
print(f"  Difference: {abs(glucose_uptake - pyr_formation)}")

print(f"\nConstraint 2 check:")
print(f"  Pyruvate formation: {pyr_formation}")
print(f"  Lactate + Biomass: {lactate_formation + biomass_formation}")
print(f"  Difference: {abs(pyr_formation - (lactate_formation + biomass_formation))}")


Current measurements:
  Glucose_uptake: 10
  Pyr_formation: 10
  Lactate_formation: 6
  Biomass_formation: 4

Constraint 1 check:
  Glucose uptake: 10
  Pyruvate formation: 10
  Difference: 0

Constraint 2 check:
  Pyruvate formation: 10
  Lactate + Biomass: 10
  Difference: 0


### 3. Set the fluxes bounds

In [33]:
# Set flux bounds
bounds = {}

# Default bounds
default_lower = -1000.0
default_upper = 1000.0

for i, reaction_name in enumerate(reaction_names):
    if flux_bounds and reaction_name in flux_bounds:
        bounds[i] = flux_bounds[reaction_name]
    else:
        bounds[i] = (default_lower, default_upper)

### 4. Setting up the flux reconciliation model

In [34]:
# Initialize Pyomo model
model = pyo.ConcreteModel()
        
# Sets
model.metabolites = pyo.Set(initialize=range(n_metabolites))
model.reactions = pyo.Set(initialize=range(n_reactions))
model.measured_reactions = pyo.Set(initialize=[
    j for j, name in enumerate(reaction_names)
    if name in measurements 
])

# Variables
# Reaction fluxes
model.v = pyo.Var(model.reactions, bounds=(-1000, 1000))

# Dual variables for optimiality conditions
model.alpha_L = pyo.Var(model.reactions, bounds=(0, 1000))    # Lower bound multipliers
model.alpha_U = pyo.Var(model.reactions, bounds=(0, 1000))    # Upper bound multipliers

# Apply flux bounds to reaction variables
for i in model.reactions:
    model.v[i].setlb(bounds[i][0])
    model.v[i].setub(bounds[i][1])
    
# # Fixed fluxes (if specified, uptake rates)
# if fixed_fluxes:
#     for reaction_name, flux_value in fixed_fluxes.items():
#         if reaction_name in reaction_names:
#             reaction_idx = reaction_names.index(reaction_name)
#             model.v[reaction_idx].fix(flux_value)
            
# Parameters
barrier_param = 1e-6

In [35]:
# Objective function: minimize weighted sum of squared deviations
def objective_rule(m):
    return sum(
        ((m.v[j] - measurements[reaction_names[j]]) /
        measurement_stdevs[reaction_names[j]])**2
        for j in m.measured_reactions
    )
model.obj = pyo.Objective(rule=objective_rule, sense=pyo.minimize)

# Mass balance constraints (stoichiometric constraints)
def mass_balance_rule(m, i):
    return sum(S[i, j] * m.v[j] for j in m.reactions) == 0
model.mass_balance = pyo.Constraint(model.metabolites, rule=mass_balance_rule)

# KKT stationarity conditions
def stationarity_rule(m, j):
    return (objective_weights[j] + m.alpha_L[j] - m.alpha_U[j] == 0)
model.stationarity = pyo.Constraint(model.reactions, rule=stationarity_rule)
    
# Complementarity constraints (relaxed with barrier parameter)
def comp_lower_rule(m, j):
    return (m.v[j] - bounds[j][0]) * m.alpha_L[j] <= barrier_param
model.comp_lower = pyo.Constraint(model.reactions, rule=comp_lower_rule)
        
def comp_upper_rule(m, j):
    return (bounds[j][1] - m.v[j]) * m.alpha_U[j] <= barrier_param
model.comp_upper = pyo.Constraint(model.reactions, rule=comp_upper_rule)


In [28]:
# Configure solver
solver_name = 'ipopt'  # You can change this to another solver if needed
solver_options = {
    'max_iter': 3000,
    'tol': 1e-8,
    'acceptable_tol': 1e-6,
}

solver = pyo.SolverFactory('ipopt')

for key, value in solver_options.items():
        solver.options[key] = value


# Solve
try:
    results = solver.solve(model, tee=True)
    
    if (results.solver.termination_condition == TerminationCondition.optimal or
        results.solver.termination_condition == TerminationCondition.locallyOptimal):
        
        # Extract results
        reconciled_fluxes = {
            reaction_names[j]: pyo.value(model.v[j])
            for j in model.reactions
        }
        
        objective_value = pyo.value(model.obj)
        
        results_dict = {
            'reconciled_reaction_fluxes': reconciled_fluxes,
            'objective_value': objective_value,
            'solver_status': 'optimal',
            'original_measurements': measurements,
            'measurement_stdevs': measurement_stdevs
        }
        
        print(f"Objective value: {objective_value}")
        print("Reconciled fluxes:")
        for reaction, flux in reconciled_fluxes.items():
            print(f"  {reaction}: {flux:.4f}")
        
    else:
        print(f"Solver terminated with condition: {results.solver.termination_condition}")
        results_dict = {'solver_status': 'failed', 'termination_condition': str(results.solver.termination_condition)}
        
except Exception as e:
    print(f"Error during solving: {str(e)}")
    results_dict = {'solver_status': 'error', 'error_message': str(e)}

Ipopt 3.11.1: max_iter=3000
tol=1e-08
acceptable_tol=1e-06


******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit http://projects.coin-or.org/Ipopt
******************************************************************************

NOTE: You are using Ipopt by default with the MUMPS linear solver.
      Other linear solvers might be more efficient (see Ipopt documentation).


This is Ipopt version 3.11.1, running with linear solver mumps.

Number of nonzeros in equality constraint Jacobian...:       13
Number of nonzeros in inequality constraint Jacobian.:       16
Number of nonzeros in Lagrangian Hessian.............:       12

Total number of variables............................:       12
                     variables with only lower bounds:        0
                var

# *E. coli* core model

In [11]:
# Add paths
sys.path.append(os.path.abspath('..'))
sys.path.append(os.path.abspath('../scripts'))

# Add directories
data_dir = "../data"

# Load model
core_model_path = os.path.join(data_dir, "raw", "GEMs", "e_coli_core.xml")
core_model = cobra.io.read_sbml_model(core_model_path)

print(f'Number of reactions for core model: {len(core_model.reactions)}')

Number of reactions for core model: 95


In [12]:
# FBA check
core_model.optimize()
print(core_model.objective.value)

0.8739215069684302
